# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata object (not as a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the mlcroissant dataset API and metadata. All references to entities use their `@id`.

**List record sets and their fields by `@id`:**

In [ ]:
# List all available record sets and their fields by @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - {field['@id']} ({field['name']})")
        print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

If you know the record set `@id`s, replace `<RS_ID>` as needed.

In [ ]:
# Get all available record set @ids
record_sets_metadata = dataset.metadata.record_sets
record_set_ids = [rs['@id'] for rs in record_sets_metadata]
print("Available Record Set @ids:", record_set_ids)

# Dictionary to store DataFrames for each record set
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"{rs_id} columns: {df.columns.tolist()}")
        print(df.head(2))
        print()
    else:
        print(f"No records found for {rs_id}\n")
        dataframes[rs_id] = pd.DataFrame()

# For demonstration, pick the first non-empty record set
main_rs_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_rs_id = rid
        break
if main_rs_id:
    print(f"Main record set for analysis: {main_rs_id}")
    print(dataframes[main_rs_id].head())
else:
    print("No non-empty record sets found to analyze.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps such as filtering, normalizing, and grouping. All columns are referenced by their `@id` as in the Croissant schema.

**Please update the field IDs as needed after running the previous cell to see field names.**

In [ ]:
# Example: pick a numeric field and a group field by their @id (column name)
# Replace these with actual IDs visible from the previous cell's DataFrame columns
record_set_id = main_rs_id  # The main record set chosen above
df = dataframes[record_set_id]

# If the dataset is non-empty, proceed with EDA
if not df.empty:
    # You must set these IDs according to your dataset's schema fields
    # For demo, we try to infer likely numeric fields
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi' and not df[col].isnull().all()]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field}")
    else:
        print("No numeric field found.")
        numeric_field = None

    group_candidates = [col for col in df.columns if df[col].dtype == object]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        print(f"Selected group field: {group_field}")

    # Now filter, normalize, and group (if possible)
    if numeric_field:
        threshold = df[numeric_field].mean()  # Set threshold as mean for this example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (showing means):")
            print(grouped_df.head())
else:
    print("No data available for EDA analysis.")

## 5. Visualization
Visualize the distribution of a numeric field and the mean by group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to access, load, and explore a dataset described using the Croissant schema and retrieved with `mlcroissant`, using entity `@id`s for all references. You can identify dataset structure, extract tabular content, apply EDA techniques, and visualize your fields for deeper analytical understanding. Adapt the field IDs and data logic as appropriate for your dataset!